# Silver - Limpieza y estructura de transacciones BYMA

Toma Bronze, tipa correctamente, y resuelve las variantes de liquidación de símbolo
(ej. AL30 vs AL30D: mismo instrumento base, distinta moneda de liquidación).
Solo pasan a Silver los registros con `flag_calidad_ok = true` — los marcados quedan
documentados en Bronze como fuente de trazabilidad, no se replican en capas posteriores.

In [0]:
from pyspark.sql import functions as F
from databricks.sdk.runtime import spark, dbutils, display

CATALOGO = "workspace"
SCHEMA = "default"
TABLA_BRONZE = f"{CATALOGO}.{SCHEMA}.bronze_transacciones"
TABLA_SILVER = f"{CATALOGO}.{SCHEMA}.silver_transacciones"

In [0]:
df_bronze = spark.table(TABLA_BRONZE).filter(F.col("flag_calidad_ok") == True)

print(f"Registros que pasan a Silver: {df_bronze.count()}")

Registros que pasan a Silver: 87502


## Resolución de variantes de símbolo

Un símbolo con sufijo `D` (ej. `AL30D`) es el mismo instrumento base que su versión sin
sufijo (`AL30`), pero liquidado en dólar cable en vez de pesos. Se separa en dos columnas:
`simbolo_base` (para agrupar el mismo instrumento sin importar la liquidación) y
`liquidacion` (ARS o USD).

In [0]:
df_silver = (
    df_bronze
    .withColumn(
        "simbolo_base",
        F.when(
            F.col("simbolo_titulo").endswith("D"),
            F.expr("substring(simbolo_titulo, 1, length(simbolo_titulo) - 1)")
        ).otherwise(F.col("simbolo_titulo"))
    )
    .withColumn(
        "liquidacion",
        F.when(F.col("simbolo_titulo").endswith("D"), F.lit("USD")).otherwise(F.col("moneda"))
    )
    .withColumn("cantidad", F.col("cantidad").cast("long"))
    .withColumn("precio", F.col("precio").cast("decimal(18,4)"))
    .select(
        "id_transaccion",
        "fecha",
        "fecha_particion",
        "tipoTran",
        "id_cliente",
        "descripcion_titulo",
        "simbolo_titulo",
        "simbolo_base",
        "moneda",
        "liquidacion",
        "cantidad",
        "precio",
        "origen",
        "fecha_ingesta"
    )
)

## Verificación del mapeo de símbolos

In [0]:
df_silver.select("simbolo_titulo", "simbolo_base", "liquidacion").distinct().orderBy("simbolo_base").show(20, truncate=False)

+--------------+-------------+-----------+
|simbolo_titulo|simbolo_base |liquidacion|
+--------------+-------------+-----------+
|#MAV150560106 |#MAV150560106|ARS        |
|A3            |A3           |ARS        |
|AAL           |AAL          |ARS        |
|AALD          |AAL          |USD        |
|AAP           |AAP          |ARS        |
|AAPLD         |AAPL         |USD        |
|AAPL          |AAPL         |ARS        |
|AAPLC         |AAPLC        |USD        |
|ABBVD         |ABBV         |USD        |
|ABBV          |ABBV         |ARS        |
|ABEVD         |ABEV         |USD        |
|ABEV          |ABEV         |ARS        |
|ABEV3         |ABEV3        |ARS        |
|ABNB          |ABNB         |ARS        |
|ABNBD         |ABNB         |USD        |
|ABT           |ABT          |ARS        |
|ABTD          |ABT          |USD        |
|ACND          |ACN          |USD        |
|ACN           |ACN          |ARS        |
|ACWID         |ACWI         |USD        |
+----------

## Escritura idempotente a Delta

In [0]:
fechas_a_cargar = [r["fecha_particion"] for r in df_silver.select("fecha_particion").distinct().collect()]
tabla_existe = spark.catalog.tableExists(TABLA_SILVER)

if not tabla_existe:
    (
        df_silver.write
        .format("delta")
        .partitionBy("fecha_particion")
        .mode("overwrite")
        .saveAsTable(TABLA_SILVER)
    )
    print(f"Tabla {TABLA_SILVER} creada con {df_silver.count()} registros.")
else:
    fecha_min = min(fechas_a_cargar)
    fecha_max = max(fechas_a_cargar)
    (
        df_silver.write
        .format("delta")
        .option("replaceWhere", f"fecha_particion >= '{fecha_min}' AND fecha_particion <= '{fecha_max}'")
        .mode("overwrite")
        .saveAsTable(TABLA_SILVER)
    )
    print(f"Particiones entre {fecha_min} y {fecha_max} reemplazadas en {TABLA_SILVER}.")

Particiones entre 2026-01-02 y 2026-03-13 reemplazadas en workspace.default.silver_transacciones.
